# Chapter 32 bridge: convolution and pooling against autograd

Chapter 32 derives the convolution gradient and the max-pooling gradient by hand. Both are checked here against `torch.nn.functional.conv2d` and `max_pool2d`, on the chapter's own filters and images. The pooling check also examines what each implementation does when two values inside a window are tied.

Nothing here is reimplemented. The chapter's own file is executed, its own weights and data are handed to PyTorch, and the chapter's hand-derived gradients are compared against autograd. Tolerances are stated per check and are **relative** to the size of the quantity being compared, because an absolute threshold means nothing without a scale.

Run order: top to bottom, from a fresh kernel. Requires `requirements-bridges.txt` on top of the book's own `requirements.txt`.

In [1]:
import os, sys, numpy as np, torch
torch.set_default_dtype(torch.float64)          # match NumPy's float64 exactly
CH = os.path.join("..", "code", "ch32")
os.chdir(CH) if os.path.basename(os.getcwd()) != "ch32" else None
def run(name):
    exec(open(name, encoding="utf-8").read(), globals())
run("_lib.py")
print("chapter:", os.path.basename(os.getcwd()), "| torch", torch.__version__, "| numpy", np.__version__)


chapter: ch32 | torch 2.14.0 | numpy 2.4.4


In [2]:
def report(name, ours, theirs, tol=1e-9):
    a = np.asarray(ours, dtype=float); b = np.asarray(theirs, dtype=float)
    denom = max(np.abs(b).max(), 1e-300)
    absd = np.abs(a - b).max(); rel = absd / denom
    ok = rel <= tol
    RESULTS.append(dict(check=name, max_abs=float(absd), max_rel=float(rel),
                        scale=float(denom), tol=tol, passed=bool(ok)))
    print(f"{'PASS' if ok else 'FAIL'}  {name:52s} max|diff| {absd:.3e}   "
          f"relative {rel:.2e}   (tolerance {tol:g})")
    return ok
RESULTS = []
MEASUREMENTS = []      # reported, never asserted: these have no single right answer


In [3]:
run("c1.py"); run("c2.py"); run("c3.py"); run("c4.py"); run("c5.py")   # c5 defines the vectorized conv and pool

input image      (8, 8)
3x3 filter        (3, 3)
output feature map (6, 6)   <- shrinks by kernel_size - 1

one output value, by hand, at position (2, 2):
  image patch:
[[0.94 0.12 0.  ]
 [0.75 0.   0.  ]
 [0.5  0.   0.  ]]
  patch . filter (elementwise, summed) = 2.1875
  convolve2d agrees: 2.1875
architecture                  parameters
fully connected, 32 hidden         2,048
conv, 4 filters of 3x3                36
ratio: 57x fewer parameters in the conv layer

at a 8x8 image:
  fully connected, 32 hidden:      2,048 parameters
  conv, 4 filters of 3x3:             36 parameters  (unchanged)

at a 32x32 image:
  fully connected, 32 hidden:     32,768 parameters
  conv, 4 filters of 3x3:             36 parameters  (unchanged)

at a 256x256 image:
  fully connected, 32 hidden:  2,097,152 parameters
  conv, 4 filters of 3x3:             36 parameters  (unchanged)
feature map     (6, 6)
after 2x2 pool  (3, 3)

after shifting the image by one pixel:
  mean change in the raw feature map

     30       1.3966         0.6861


     60       0.8174         0.8222


     90       0.5152         0.8611


    120       0.3675         0.8972


    150       0.2870         0.9167

final CNN test accuracy: 0.9167


### Convolution forward

In [4]:
imgs = Xtr_img[:8]
filters = np.random.default_rng(32).normal(0, 0.3, (4, 3, 3))
feats, windows = conv_forward(imgs, filters)
timgs = torch.tensor(imgs).unsqueeze(1)
tfil = torch.tensor(filters, requires_grad=True)          # keep it a leaf so .grad is populated
tfeat = torch.nn.functional.conv2d(timgs, tfil.unsqueeze(1))
report("convolution forward", feats, tfeat.detach().numpy(), 1e-12)

PASS  convolution forward                                  max|diff| 4.441e-16   relative 2.56e-16   (tolerance 1e-12)


np.True_

### Convolution backward

In [5]:
dout = np.random.default_rng(1).normal(0, 1, feats.shape)
_, dfil = conv_backward(dout, windows, filters)
tfeat.backward(torch.tensor(dout))
report("convolution backward: dfilters", dfil, tfil.grad.numpy())

PASS  convolution backward: dfilters                       max|diff| 8.882e-15   relative 5.18e-16   (tolerance 1e-09)


np.True_

### Max pooling, forward and backward

In [6]:
pooled, mask = pool_forward(feats)
tf2 = torch.tensor(feats, requires_grad=True)
tp = torch.nn.functional.max_pool2d(tf2, 2)
report("max pooling forward", pooled, tp.detach().numpy(), 1e-12)
dp = np.random.default_rng(2).normal(0, 1, pooled.shape)
dfeats = pool_backward(dp, mask)
tp.backward(torch.tensor(dp))
tg = tf2.grad.numpy()

# A pooling window whose maximum is unique has ONE correct gradient and both
# implementations must agree exactly. A window with a tied maximum has no single right
# answer: the chapter's mask feeds every tied cell, torch feeds one. Split the check so
# the exact part stays exact and the ambiguous part is reported rather than averaged in.
n_, nf_, h_, w_ = feats.shape
blk = feats.reshape(n_, nf_, h_//2, 2, w_//2, 2)
mx = blk.max(axis=(3, 5), keepdims=True)
n_max = (blk == mx).sum(axis=(3, 5))                       # how many cells hold the max
tied = (n_max > 1)                                         # (n, nf, h/2, w/2)
tied_cells = np.repeat(np.repeat(tied, 2, axis=2), 2, axis=3)
print(f"pooling windows in this batch: {tied.size:,}   with a tied maximum: {int(tied.sum()):,} "
      f"({100*tied.mean():.2f}%)")
report("max pooling backward, windows with a unique maximum",
       dfeats[~tied_cells], tg[~tied_cells])
diff_tied = np.abs(dfeats[tied_cells] - tg[tied_cells])
print(f"tied windows: {int(tied_cells.sum()):,} cells, max |difference| {diff_tied.max():.4f} "
      f"-- the two tie-breaking conventions, not an error")
MEASUREMENTS.append(dict(measurement="max pooling backward, tied windows",
                         max_abs=float(diff_tied.max()) if diff_tied.size else 0.0))

PASS  max pooling forward                                  max|diff| 0.000e+00   relative 0.00e+00   (tolerance 1e-12)
pooling windows in this batch: 288   with a tied maximum: 2 (0.69%)
PASS  max pooling backward, windows with a unique maximum  max|diff| 0.000e+00   relative 0.00e+00   (tolerance 1e-09)
tied windows: 8 cells, max |difference| 1.7224 -- the two tie-breaking conventions, not an error


### What happens at a tie
When two values inside a pooling window are equal, the gradient has to go somewhere. The chapter's mask sends it to **every** tied position; PyTorch sends it to **one**. Neither is wrong, and on real-valued feature maps ties essentially never occur -- but on a hand-made tie the two disagree, and that is worth seeing rather than hiding.

In [7]:
tie = np.zeros((1, 1, 2, 2)); tie[0, 0] = [[5.0, 5.0], [1.0, 2.0]]
p_np, m_np = pool_forward(tie)
tt = torch.tensor(tie, requires_grad=True)
tpt = torch.nn.functional.max_pool2d(tt, 2)
tpt.backward(torch.ones_like(tpt))
print("chapter's gradient at the tie:\n", pool_backward(np.ones_like(p_np), m_np)[0, 0])
print("torch's gradient at the tie:\n", tt.grad.numpy()[0, 0])
n_, nf_, h_, w_ = feats.shape
blocks = feats.reshape(n_, nf_, h_//2, 2, w_//2, 2).transpose(0,1,2,4,3,5).reshape(-1, 4)
frac = float(np.mean([(np.sort(b)[-1] == np.sort(b)[-2]) for b in blocks]))
print(f"\nshare of real pooling windows in this batch with a tied maximum: {100*frac:.2f}%")
MEASUREMENTS.append(dict(measurement="max-pool tie: the two conventions differ by design"))

chapter's gradient at the tie:
 [[1. 1.]
 [0. 0.]]
torch's gradient at the tie:
 [[1. 0.]
 [0. 0.]]

share of real pooling windows in this batch with a tied maximum: 0.69%


In [8]:
import json
n_pass = sum(1 for r in RESULTS if r["passed"])
print(f"\n{n_pass} of {len(RESULTS)} ASSERTED checks passed")
if MEASUREMENTS:
    print(f"{len(MEASUREMENTS)} reported measurement(s), not asserted:")
    for m in MEASUREMENTS: print("   ", m)
print(json.dumps(dict(checks=RESULTS, measurements=MEASUREMENTS), indent=1))
assert n_pass == len(RESULTS), "a gradient check failed"



4 of 4 ASSERTED checks passed
2 reported measurement(s), not asserted:
    {'measurement': 'max pooling backward, tied windows', 'max_abs': 1.7224233963662954}
    {'measurement': 'max-pool tie: the two conventions differ by design'}
{
 "checks": [
  {
   "check": "convolution forward",
   "max_abs": 4.440892098500626e-16,
   "max_rel": 2.5575675775730126e-16,
   "scale": 1.7363733171479998,
   "tol": 1e-12,
   "passed": true
  },
  {
   "check": "convolution backward: dfilters",
   "max_abs": 8.881784197001252e-15,
   "max_rel": 5.183665744596186e-16,
   "scale": 17.134176149880503,
   "tol": 1e-09,
   "passed": true
  },
  {
   "check": "max pooling forward",
   "max_abs": 0.0,
   "max_rel": 0.0,
   "scale": 1.7363733171479996,
   "tol": 1e-12,
   "passed": true
  },
  {
   "check": "max pooling backward, windows with a unique maximum",
   "max_abs": 0.0,
   "max_rel": 0.0,
   "scale": 2.845645074509526,
   "tol": 1e-09,
   "passed": true
  }
 ],
 "measurements": [
  {
   "measureme